# MGMT298D: Science and Strategy of AI
## Assignment 8 - Building with Generative AI APIs
### Application: Creating AI-Powered Features

---

**Instructions:** Complete the exercises by filling in the `???` placeholders and answering the questions. Run all code cells in order.

## Setup and API Configuration

In [ ]:
!pip install -q -U google-generativeai

import google.generativeai as genai
from google.colab import userdata
import json
import time

api_key = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-1.5-flash-latest')

def generate(prompt, temperature=1.0, top_k=40, top_p=0.95):
    """Generate text with configurable parameters."""
    response = model.generate_content(
        prompt,
        generation_config={
            'temperature': temperature,
            'top_k': top_k,
            'top_p': top_p
        }
    )
    return response.text

print(generate("Say 'API connected!' in exactly 2 words.", temperature=0.1))

## Part 1: Understanding Temperature

Temperature controls the randomness of output. Let's see how it affects generation.

In [ ]:
prompt = "Complete this sentence creatively: The robot walked into the coffee shop and"

print("=== Temperature 0.2 (Low - Deterministic) ===")
for i in range(3):
    print(f"{i+1}. {generate(prompt, temperature=0.2)}")
    print()

print("\n=== Temperature 1.5 (High - Creative) ===")
for i in range(3):
    print(f"{i+1}. {generate(prompt, temperature=1.5)}")
    print()

In [ ]:
# ============================================================
# EXERCISE: Find the right temperature for different tasks
# ============================================================

# Task 1: Generate a factual summary (should be consistent)
fact_prompt = "Summarize what machine learning is in one sentence."
FACT_TEMP = ???  # Try values between 0.1 and 0.5

print("Factual task (low temp recommended):")
for i in range(3):
    print(f"{i+1}. {generate(fact_prompt, temperature=FACT_TEMP)}")

print("\n" + "="*50 + "\n")

# Task 2: Generate creative product names (should vary)
creative_prompt = "Suggest a creative name for a new energy drink:"
CREATIVE_TEMP = ???  # Try values between 1.0 and 1.8

print("Creative task (high temp recommended):")
for i in range(3):
    print(f"{i+1}. {generate(creative_prompt, temperature=CREATIVE_TEMP)}")

## Part 2: Top-K and Top-P Sampling

These parameters control which tokens the model considers when generating.

In [ ]:
prompt = "List 5 unusual pizza toppings:"

print("=== Top-K = 5 (Very restricted choices) ===")
print(generate(prompt, temperature=1.0, top_k=5))

print("\n=== Top-K = 100 (Many choices) ===")
print(generate(prompt, temperature=1.0, top_k=100))

In [ ]:
print("=== Top-P = 0.5 (Only most likely tokens) ===")
print(generate(prompt, temperature=1.0, top_p=0.5))

print("\n=== Top-P = 0.99 (Almost all tokens considered) ===")
print(generate(prompt, temperature=1.0, top_p=0.99))

## Part 3: Building a Structured Output Generator

For real applications, you often need the API to return structured data.

In [ ]:
def analyze_review(review_text):
    """Analyze a customer review and return structured JSON."""
    prompt = f"""
Analyze this customer review and return ONLY valid JSON:

Review: "{review_text}"

Return this exact format:
{{
  "sentiment": "positive" or "negative" or "mixed",
  "rating_estimate": 1-5,
  "key_topics": ["topic1", "topic2"],
  "action_needed": true or false,
  "summary": "one sentence summary"
}}
"""
    response = generate(prompt, temperature=0.2)  # Low temp for consistency
    
    # Parse JSON
    try:
        clean = response.replace('```json', '').replace('```', '').strip()
        return json.loads(clean)
    except:
        return {"error": "Failed to parse", "raw": response}

# Test with sample reviews
reviews = [
    "Amazing product! Fast shipping and exactly what I needed. Will buy again!",
    "The item broke after 2 days. Very disappointed. Requesting refund.",
    "It's okay. Does the job but nothing special. Packaging was nice though."
]

for review in reviews:
    print(f"Review: {review[:50]}...")
    result = analyze_review(review)
    print(f"Analysis: {json.dumps(result, indent=2)}\n")

In [ ]:
# ============================================================
# EXERCISE: Build your own structured generator
# ============================================================

def extract_meeting_info(meeting_notes):
    """Extract key information from meeting notes."""
    prompt = f"""
Extract information from these meeting notes and return ONLY valid JSON:

Notes: "{meeting_notes}"

Return this exact format:
{{
  "attendees": ["name1", "name2"],
  "decisions": ["decision1", "decision2"],
  "action_items": ["item1", "item2"],
  "next_meeting": "date or null"
}}
"""
    response = generate(prompt, temperature=???)  # Fill in appropriate temperature
    
    try:
        clean = response.replace('```json', '').replace('```', '').strip()
        return json.loads(clean)
    except:
        return {"error": "Failed to parse"}

# Test your function
sample_notes = """
Meeting with Sarah and Mike on Monday. Decided to launch the new feature next week.
Sarah will prepare the marketing materials. Mike will coordinate with engineering.
Follow-up scheduled for Friday at 2pm.
"""

result = extract_meeting_info(sample_notes)
print(json.dumps(result, indent=2))

## Part 4: Building a Simple Chatbot with Memory

In [ ]:
class SimpleChatbot:
    """A chatbot that remembers conversation history."""
    
    def __init__(self, system_prompt, temperature=0.7):
        self.system_prompt = system_prompt
        self.temperature = temperature
        self.history = []
    
    def chat(self, user_message):
        # Build context from history
        context = f"System: {self.system_prompt}\n\n"
        for msg in self.history[-6:]:  # Keep last 6 messages for context
            context += f"{msg['role']}: {msg['content']}\n"
        context += f"User: {user_message}\nAssistant:"
        
        # Generate response
        response = generate(context, temperature=self.temperature)
        
        # Update history
        self.history.append({"role": "User", "content": user_message})
        self.history.append({"role": "Assistant", "content": response})
        
        return response
    
    def reset(self):
        self.history = []

# Create a customer service bot
bot = SimpleChatbot(
    system_prompt="You are a helpful customer service agent for TechGadgets Inc. Be friendly and concise.",
    temperature=0.5
)

# Simulate a conversation
messages = [
    "Hi, I bought a laptop last week and it's not turning on.",
    "I've tried holding the power button for 10 seconds but nothing happens.",
    "Yes, it's plugged in. The charging light is on."
]

for msg in messages:
    print(f"User: {msg}")
    response = bot.chat(msg)
    print(f"Bot: {response}\n")

## Part 5: Batch Processing for Business Applications

In [ ]:
import pandas as pd

# Sample customer feedback data
feedback_data = pd.DataFrame({
    'customer_id': ['C001', 'C002', 'C003', 'C004', 'C005'],
    'feedback': [
        "Love the product! Works perfectly.",
        "Shipping took forever. Product is okay.",
        "Terrible experience. Item was damaged.",
        "Good value for money. Recommend!",
        "Not what I expected. Quality is poor."
    ]
})

def batch_analyze(df, text_column):
    """Analyze a batch of text entries."""
    results = []
    
    for idx, row in df.iterrows():
        prompt = f"""
Classify this feedback. Return ONLY one word: POSITIVE, NEGATIVE, or NEUTRAL

Feedback: "{row[text_column]}"
"""
        sentiment = generate(prompt, temperature=0.1).strip().upper()
        results.append(sentiment)
        time.sleep(0.3)  # Rate limiting
    
    df['sentiment'] = results
    return df

# Run batch analysis
analyzed_df = batch_analyze(feedback_data.copy(), 'feedback')
print(analyzed_df)

# Summary
print("\nSentiment Distribution:")
print(analyzed_df['sentiment'].value_counts())

In [ ]:
# ============================================================
# EXERCISE: Create your own batch processor
# ============================================================

# Sample product data
products = pd.DataFrame({
    'product_name': ['Smart Watch X1', 'Wireless Earbuds Pro', 'Laptop Stand Deluxe'],
    'features': [
        'Heart rate monitor, GPS, water resistant, 5-day battery',
        'Noise cancellation, 24hr battery, touch controls',
        'Adjustable height, aluminum, cable management'
    ]
})

def generate_descriptions(df):
    """Generate marketing descriptions for products."""
    descriptions = []
    
    for idx, row in df.iterrows():
        prompt = ???  # Write a prompt to generate a 2-sentence marketing description
        
        description = generate(prompt, temperature=???)
        descriptions.append(description.strip())
        time.sleep(0.3)
    
    df['description'] = descriptions
    return df

# Run your batch processor
products_with_descriptions = generate_descriptions(products.copy())
for _, row in products_with_descriptions.iterrows():
    print(f"Product: {row['product_name']}")
    print(f"Description: {row['description']}\n")

## Part 6: Error Handling and Production Considerations

In [ ]:
def robust_generate(prompt, max_retries=3, **kwargs):
    """Production-ready API call with retries and error handling."""
    for attempt in range(max_retries):
        try:
            response = model.generate_content(
                prompt,
                generation_config=kwargs
            )
            return {"success": True, "text": response.text, "attempts": attempt + 1}
        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # Exponential backoff
                print(f"Attempt {attempt + 1} failed. Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                return {"success": False, "error": str(e), "attempts": attempt + 1}

# Test the robust function
result = robust_generate("What is 2+2?", temperature=0.1)
print(f"Success: {result['success']}")
print(f"Response: {result.get('text', result.get('error'))}")
print(f"Attempts needed: {result['attempts']}")

---
## Questions

### Question 1: Temperature Selection

**Q1:** Based on your experiments, what temperature would you recommend for each of these business applications? Explain your reasoning.
- (a) Generating legal contract summaries
- (b) Writing social media posts
- (c) Answering customer FAQ questions

*Your answer:*


---
### Question 2: Top-K vs Top-P

**Q2:** Explain the difference between Top-K and Top-P sampling in your own words. When might you want to use a low Top-K value? When might Top-P be more useful than Top-K?

*Your answer:*


---
### Question 3: Structured Output Reliability

**Q3:** When building applications that require JSON output from an LLM, what can go wrong? What strategies did we use in this assignment to improve reliability, and what additional safeguards might you add for a production system?

*Your answer:*


---
### Question 4: Cost and Scale Considerations

**Q4:** A company wants to use the API to analyze 100,000 customer reviews per day. What are the main challenges they'll face (think about: cost, rate limits, latency, error handling)? How would you design the system to handle this scale?

*Your answer:*


---
### Question 5: Build vs Buy

**Q5:** Your startup needs to add AI features to your product. Compare these three approaches:
- (a) Using a commercial API like Gemini or GPT-4
- (b) Fine-tuning an open-source model
- (c) Building a custom model from scratch

What factors would influence your decision? When is each approach most appropriate?

*Your answer:*
